# Runde 1b · Vorbereitung prüfen und fortsetzen · CPU, Konto A

**In Konto A auf einer CPU-Laufzeit öffnen, dann „Alle ausführen“.** Vorher alte
Runde-1b-Notebooks in beiden Konten stoppen. `dsse_runs_B` muss in Konto A als
Verknüpfung mit Bearbeiterrecht erreichbar sein.

Dieses Notebook lädt den aktuellen Vorbereitungscode in einen neuen temporären
Ordner unter `/content`. Es prüft beziehungsweise vervollständigt dieselben
Variantenordner wie HTQ_START. Eine vollständig vorbereitete Ablage von Konto A
wird wiederverwendet. Originalergebnisse und vorhandene Sperren bleiben erhalten.

Die letzte Zelle speichert die vollständige Vorbereitungsausgabe einschließlich
Fehler in Konto A unter `dsse_runs/htq_prep_check.log`. Den aktuellen Laufstatus
mit UTC-Zeit enthält `dsse_runs/htq_prep_check_status.json`. So bleibt die Ursache
lesbar, wenn Colab die Zellenausgabe auf dem Telefon nicht anzeigt. Im Notebook
erscheinen nur kurze Statusmeldungen. Die vorherigen Zellen zum Verbinden von
Drive und Laden des Codes liegen außerhalb dieses Diagnoseprotokolls.
Dieses Notebook lädt keine Modelle und startet keine GPU-Messung.

Nach erfolgreicher Vorbereitung beider Konten **HTQ_START auf der A100** ausführen.
Bei einem Fehler das Protokoll in Drive öffnen. Sperren oder Variantenordner
nicht auf Verdacht löschen.


In [ ]:
import os, sys, json, subprocess, tempfile
from google.colab import drive
drive.mount("/content/drive")
MYDRIVE = "/content/drive/MyDrive"
SOURCE_A = f"{MYDRIVE}/dsse_runs"
SOURCE_B = f"{MYDRIVE}/dsse_runs_B"
assert os.path.isdir(SOURCE_A), "Konto A: dsse_runs fehlt."
assert os.path.isdir(SOURCE_B), "Konto B: Verknüpfung dsse_runs_B fehlt; Freigabe und Bearbeiterrecht prüfen."
B_RUNS = os.path.join(SOURCE_B, "variants/htq_v2")
BRANCH = "codex/r1b-htq-continuation"
NUR_SELBSTTEST = False
_variant_keys = ("R1B_H_POLICY", "R1B_RUNS_B", "R1B_XFER_A", "R1B_SOURCE_RUNS")
TEST_ENV = {k: v for k, v in os.environ.items() if k not in _variant_keys}


In [ ]:
# Neue Arbeitskopie: vorhandene /content/DSSE-Repositories werden nicht verändert.
REPO = tempfile.mkdtemp(prefix="dsse_htq_prep_", dir="/content")
print("Aktuellen Vorbereitungscode laden:", REPO, flush=True)
subprocess.run(["git", "clone", "-q", "--filter=blob:none", "--sparse", "-b", BRANCH,
                "https://github.com/Erikiss/DSSE", REPO], check=True, env=TEST_ENV)
subprocess.run(["git", "-C", REPO, "sparse-checkout", "set", "analysis/ju_space",
                "results/ju_space/plans"], check=True, env=TEST_ENV)
COMMIT = subprocess.run(["git", "-C", REPO, "rev-parse", "HEAD"], capture_output=True,
                        text=True, check=True, env=TEST_ENV).stdout.strip()
R1B = f"{REPO}/analysis/ju_space/round1b"
print("Vorbereitungscode:", COMMIT, flush=True)


In [ ]:
# Dauerhaftes Diagnoseprotokoll: diese letzte Zelle enthält die Vorbereitung.
import contextlib, json, os, traceback, uuid
from datetime import datetime, timezone
_prep_program = '# Varianten-Bootstrap / Prüfung vor jedem realen Kopier- oder Hauptlauf.\n# Variantenwerkzeug: Ausgabe und Fehler mit Konto sichtbar streamen.\nimport textwrap\nVARIANT_TOOL = f"{R1B}/r1b_htq_variant.py"\ndef run_variant(action, path, account):\n    def show(message):\n        for line in str(message).splitlines() or [""]:\n            for wrapped in textwrap.wrap(line, width=72, replace_whitespace=False,\n                                         drop_whitespace=False, break_on_hyphens=False) or [""]:\n                print(wrapped, flush=True)\n    show(f"[Konto {account}] {action}: {path}")\n    command = [sys.executable, "-u", VARIANT_TOOL, action, path, account]\n    try:\n        process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,\n                                   text=True, bufsize=1, env=TEST_ENV)\n    except OSError as error:\n        show(f"[Konto {account}] {action}: Start fehlgeschlagen: {error}")\n        raise RuntimeError(f"Konto {account}: {action} konnte nicht starten.") from error\n    try:\n        for line in process.stdout:\n            show(line.rstrip("\\r\\n"))\n        code = process.wait()\n    except BaseException:\n        if process.poll() is None:\n            process.terminate()\n            try:\n                process.wait(timeout=10)\n            except subprocess.TimeoutExpired:\n                process.kill()\n                process.wait()\n        show(f"[Konto {account}] {action}: unterbrochen; Kindprozess beendet.")\n        raise\n    finally:\n        process.stdout.close()\n    show(f"[Konto {account}] {action}: Exit {code}")\n    if code:\n        raise RuntimeError(f"Konto {account}: {action} fehlgeschlagen (Exit {code}). Details oben.")\n    return code\n\ndef check_variant(path, account):\n    return run_variant("check", path, account)\nif not NUR_SELBSTTEST:\n    for original, account in ((SOURCE_A, "A"), (SOURCE_B, "B")):\n        run_variant("seed", original, account)\n        check_variant(os.path.join(original, "variants/htq_v2"), account)\n    paired = dict(variant="htq_v2", accounts=["A", "B"], bootstrap_commit=COMMIT)\n    for path in (os.path.join(SOURCE_A, "variants/htq_v2"), B_RUNS):\n        marker = os.path.join(path, "_htq_pair_ready.json")\n        with open(marker + ".tmp", "w") as f:\n            json.dump(paired, f, indent=1)\n        os.replace(marker + ".tmp", marker)\n    print(\'Beide Konten vorbereitet und geprüft. Jetzt HTQ_START auf der A100 ausführen.\', flush=True)\n'
_prep_source = globals().get("SOURCE_A") or "/content/drive/MyDrive/dsse_runs"
_prep_now = lambda: datetime.now(timezone.utc).isoformat()
_prep_status = dict(run_id=uuid.uuid4().hex[:12], started_utc=_prep_now(),
                    finished_utc=None, status="running", stage="compile", error=None,
                    commit=globals().get("COMMIT") if isinstance(globals().get("COMMIT"), str) else None)
try:
    if not os.path.isdir(_prep_source):
        raise NotADirectoryError("Der bestehende dsse_runs-Ordner von Konto A fehlt; Drive zuerst verbinden.")
    HTQ_PREP_LOG = os.path.join(_prep_source, "htq_prep_check.log")
    HTQ_PREP_STATUS = os.path.join(_prep_source, "htq_prep_check_status.json")
    _prep_log = open(HTQ_PREP_LOG, "a", encoding="utf-8", buffering=1)
except Exception as error:
    print("Protokoll konnte NICHT in Drive geöffnet werden:", str(error), flush=True)
    raise RuntimeError("Drive-Protokoll nicht gespeichert. Konto-A-Ablage prüfen.") from None

def _prep_save_status():
    temporary = HTQ_PREP_STATUS + ".tmp-" + _prep_status["run_id"]
    with open(temporary, "w", encoding="utf-8") as output:
        json.dump(_prep_status, output, ensure_ascii=False, indent=1)
    os.replace(temporary, HTQ_PREP_STATUS)

print("Vorbereitung läuft. Vollständige Ausgabe wird in Drive gespeichert.", flush=True)
print(HTQ_PREP_LOG, flush=True)
_prep_status_saved = False
with _prep_log:
    with contextlib.redirect_stdout(_prep_log), contextlib.redirect_stderr(_prep_log):
        print("\n=== HTQ PREP", _prep_status["run_id"], _prep_status["started_utc"], "===", flush=True)
        print("Code-Commit:", _prep_status["commit"] or "noch nicht bekannt", flush=True)
        try:
            _prep_save_status()  # Ein alter Erfolgsstatus darf nicht als aktueller gelten.
            _prep_compiled = compile(_prep_program, "<HTQ-Vorbereitung>", "exec")
            _prep_status["stage"] = "bootstrap"
            _prep_save_status()
            exec(_prep_compiled, globals())
        except BaseException as error:
            _prep_status.update(status="interrupted" if isinstance(error, KeyboardInterrupt) else "failed",
                                error_type=type(error).__name__, error=(type(error).__name__ + ": " + str(error))[:500])
            traceback.print_exc()
        else:
            _prep_status.update(status="success", stage="complete")
        finally:
            _prep_status["finished_utc"] = _prep_now()
            try:
                _prep_save_status()
                _prep_status_saved = True
            except Exception:
                print("Statusdatei konnte NICHT aktualisiert werden:", flush=True)
                traceback.print_exc()
            print("Ergebnis:", _prep_status["status"], "Lauf:", _prep_status["run_id"], flush=True)
print("Vorbereitung:", _prep_status["status"], flush=True)
print("Vollständiges Protokoll:", HTQ_PREP_LOG, flush=True)
if _prep_status_saved:
    print("Statusdatei:", HTQ_PREP_STATUS, flush=True)
else:
    print("Statusdatei NICHT aktualisiert; bitte das Protokoll lesen.", flush=True)
if _prep_status["status"] != "success":
    raise RuntimeError("Vorbereitung fehlgeschlagen. Ursache in htq_prep_check.log.") from None
print("Jetzt HTQ_START auf der A100 ausführen.", flush=True)
